In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Simulating Quantum Noise — QEC101: Lab 3 — Solutions
$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}$

---

**What You Will Do:**
* Define quantum noise channels using density matrices and Kraus operators
* Simulate noisy quantum circuits with both density matrix and trajectory-based methods in CUDA-Q
* Analyze the impact of different noise patterns on a quantum chemistry algorithm (VQE for H₂)
* Implement zero noise extrapolation as a quantum error mitigation technique
* Run noisy QEC experiments on the Steane code using the Stim simulator
* Build a noise model from dynamical simulation of a superconducting transmon qubit

**Prerequisites:**
* Python and Jupyter familiarity
* Basic quantum computing concepts (qubits, gates, measurement, braket notation) — see [Quick Start to Quantum Computing with CUDA-Q](https://github.com/NVIDIA/cuda-q-academic/tree/main/quick-start-to-quantum)
* Familiarity with CUDA-Q kernel syntax and `cudaq.sample`
* Completion of [01 — The Basics of Classical and Quantum Error Correction](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/01_QEC_Intro.ipynb)
* Completion of [02 — Stabilizers, the Shor Code, and the Steane Code](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/02_QEC_Stabilizers.ipynb)

**Key Terminology:**
* noise channel
* density matrix
* trajectory simulation
* density matrix simulation
* Kraus operator
* quantum error mitigation
* zero noise extrapolation
* circuit-level noise
* dynamical simulation
* amplitude damping
* pure state
* mixed state

**CUDA-Q Syntax:**
* [`@cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines a quantum kernel function
* [`cudaq.qvector`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.qvector) — allocates a register of qubits
* [`cudaq.sample`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample) — samples measurement outcomes
* [`cudaq.observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe) — computes expectation value of a spin operator
* [`cudaq.get_state`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.get_state) — returns the statevector or density matrix
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects simulation backend
* [`cudaq.NoiseModel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.NoiseModel) — defines a quantum noise model
* [`cudaq.SpinOperator`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.SpinOperator) — Pauli spin operator (Hamiltonian)
* [`cudaq_solvers.create_molecule`](https://nvidia.github.io/cuda-quantum/latest/api/solvers/python_api.html#cudaq_solvers.create_molecule) — builds molecular Hamiltonian from geometry
* [`cudaq.evolve`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.evolve) — runs dynamical time evolution of a quantum system

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 12px 15px 12px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900;">&#9889; GPU Required:</span>** Parts of this notebook require a GPU.

</div>

In [ ]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install CUDA-Q.

#!pip install cudaq -q
#
#!wget -q https://github.com/nvidia/cuda-q-academic/archive/refs/heads/main.zip
#!unzip -q main.zip
#!mv cuda-q-academic-main/qec101/Images ./Images

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [ ]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))

from typing import List, Optional

import numpy as np

import matplotlib.pyplot as plt

import cudaq
from cudaq import spin, operators, ScalarOperator, Schedule, ScipyZvodeIntegrator

## To install cudaq-solvers (if not already installed), uncomment and run:
## !pip install cudaq-solvers -q
## Note: cudaq-solvers requires libgfortran. If you see an ImportError, run:
## !apt-get install -y libgfortran5
import cudaq_solvers as solvers

import cupy as cp

---

## 3.1 Quantum Noise Channels

In the first lab of this series, the concept of a **noise channel** was introduced. A noise channel is a mathematical model used to describe how a quantum state is impacted by the presence of noise. 

<img src="../Images/noisy/nc.png" alt="Diagram showing a quantum state passing through a noise channel, represented as a box labeled epsilon that transforms the input state" style="width: 800px;"/>

A noise channel can correspond to application of a gate to physical qubits, a qubit's interaction with another nearby qubit, or simply the passage of time and the resulting decay of the quantum state as it interacts with anything else from the environment. QEC is a promising solution to this problem as a logically encoded quantum state can go through the noise channel, impacting each data qubit, while providing a means for the original state to be restored.

<img src="../Images/noisy/nc_qec.png" alt="Diagram showing a logical qubit encoded across multiple physical data qubits, each passing through independent noise channels, with QEC decoding restoring the original state" style="width: 1100px;"/>

However, as previous labs have emphasized, QEC is hard to implement, and the development of new QEC protocols is still an active research field.  In practice, experimental data obtained from the QPU can help measure quantities like gate fidelity and inform a noise model which captures all of the noise channels present in the device. This noise model can then be used to simulate data that emulates the performance of the QPU. 

There are many practical benefits to this that will be explored in this lab. A recent example of this is [NVIDIA's work with QuEra](https://developer.nvidia.com/blog/nvidia-and-quera-decode-quantum-errors-with-ai/) to build an AI decoder.  Training this model required a massive amount of data which could be obtained efficiently via simulation.  Noisy circuit simulation allowed for millions of syndromes to be obtained with their associated errors, something not possible to do with experimental data. 


### The Density Matrix

Before discussing some of the ways to simulate noise, it is necessary to take a step back and consider representation of a quantum state using the **density matrix**.  The density matrix ($\rho$) is a mathematical object that completely describes a quantum state and has the following properties. 

1. Its trace (sum of the diagonal elements) is equal to 1.
2. It is Hermitian: $\rho = \rho ^{\dagger}$
3. It is positive semi-definite. (All eigenvalues are positive.)

If a quantum system is in one of any quantum states $\ket{\psi_i}$ with probability $p_i$, then the density matrix is defined as a linear combination of outer products of those states with probability coefficients:

$$\rho = \sum_i p_i \ket{\psi_i}\bra{\psi_i} $$



<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 1:</span>**

Use CUDA-Q’s `get_state` function and the density matrix simulator (more on that later) to produce any three qubit density matrix. Write code to check that the three properties listed above are met. Make sure to set tolerances on these checks so that, for example, an eigenvalue of zero is not wrongfully flagged as `-1.2e-20`.

</div>

In [ ]:
# EXERCISE 1
cudaq.set_target("density-matrix-cpu")

@cudaq.kernel
def test():
    reg = cudaq.qvector(3)
    h(reg)

print("State vector:")
print(cudaq.get_state(test))

# get density matrix
rho = np.array(cudaq.get_state(test))

# Compute Trace
print("\\nTrace(rho) =", np.trace(rho))

# Check Hermitian
print("Hermitian?", np.allclose(rho, rho.conj().T, atol=1e-12))

# Check positive semi-definite
eigs, _ = np.linalg.eig(rho)
eigs[np.abs(eigs) < 1e-12] = 0
print("Eigenvalues:", eigs)


Statevectors correspond to **pure states**, while the **density matrix** can describe **mixed states**, that is an overall state composed of a combination of pure states.

A state is considered pure if the trace of $\rho^2$ is equal to 1.

This can be a bit confusing because a pure state can actually be a superposition state and a mixed state can be a combination of two states that do not describe superpositions.  The following exercise will make this more clear.


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 2:</span>**

Consider the density matrix $\rho = \frac{1}{2}\ket{00}\bra{00} + \frac{1}{2}\ket{11}\bra{11}$.

Using CUDA-Q build kernels for the $\ket{00}$ state and the $\ket{11}$ state, using these kernels and the `get_state` command define the density matrix $\rho$, and compute trace($\rho^2$). Is the state pure?

</div>

In [ ]:
# EXERCISE 2
@cudaq.kernel
def zeros():
    reg = cudaq.qvector(2)

@cudaq.kernel
def ones():
    reg = cudaq.qvector(2)
    x(reg)

rho = (.5*np.array(cudaq.get_state(zeros)) + .5*np.array(cudaq.get_state(ones))) 
print("Density Matrix")
print(rho)
rho_squared = rho@rho
print("Density Matrix Squared")
print(rho_squared)
print("Trace of Density Matrix Squared")
print(np.trace(rho_squared))

Now, code a Bell state and do the same thing with its density matrix.  Is it a pure state?

In [ ]:
@cudaq.kernel
def bell():
    reg = cudaq.qvector(2)
    
    h(reg[0])
    x.ctrl(reg[0],reg[1])

print("Density Matrix")
print(np.array(cudaq.get_state(bell)))
print("Density Matrix Squared")
print(np.array(cudaq.get_state(bell))@np.array(cudaq.get_state(bell)))
print("Trace of Density Matrix Squared")
print(np.trace(np.array(cudaq.get_state(bell))))

A mixed state means that there is classical uncertainty about which quantum state defines the system, even if both quantum states are deterministic like $\ket{00}$ and $\ket{11}$.   However, a Bell state is pure, meaning that the overall quantum state is known with certainty, even if the state describes a superposition with inherent uncertainty.  Another key term is completely mixed state, which refers to a density matrix where all of the eigenvalues are the same, meaning the density matrix describes the state with the theoretical maximum of uncertainty. 

### Kraus Operators

Now, why the business about density matrices?  The answer is that a noise channel needs to be an effective model that can generalize to impact mixed states. In fact, many noise channels will produce a mixed state from a pure state.

Mathematically this is done with **Kraus operators** ($K_i$) that evolve the density matrix as the state proceeds through a noisy channel $\epsilon$.

$$ \epsilon(\rho) = \sum_i K_i\rho K_i^{\dagger} $$

**Kraus operators** have the condition that $ \sum_i K_i K_i^{\dagger} =1 $ so the trace of the density matrix is preserved.

For example, a valid set of operators is $K_0 = \sqrt{1-p} I $ and $K_1 = \sqrt{p}X$ which will perform a bitflip error with probability $p$ and apply the identity (no change) with probability $1-p$. Let's apply this to the density matrix, $\rho_0$, for the $\ket{0}$ state. The result becomes $ \epsilon(\rho_0) = (1-p)I\rho_0 I + pX\rho_0 X $. Notice the result is now mixed state. 

The table below summarizes some of the channels included in CUDA-Q which you will use in later exercises.  Notice too, that each noise channel can be geometrically represented as a deformation of the Bloch sphere.


<img src="../Images/noisy/channels.png" alt="Table of quantum noise channels showing the Kraus operators, Bloch sphere deformations, and CUDA-Q class names for bit-flip, phase-flip, depolarization, and amplitude damping channels" style="width: 800px;"/>

 

By applying any number of Kraus operators to the density matrix, it is possible to evolve it and sample the resulting state to determine how noise impacts the output. This is easily accomplished in CUDA-Q with the `density-matrix-cpu` backend.  You can then build a noise model consisting of noisy channels applied to specific gate operations with select probabilities. The exercise below will get you started with the syntax.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 3:</span>**

You will be using CUDA-Q’s built in noise channel tools throughout this lab.  Get a sense for how it works by building a two qubit kernel and perform an $X$ operation on each qubit.  Edit the code block below to build a noise model consisting of two bitflip channels with probabilities of .10 and .25 on the $X$ gate for qubit 0 and 1, respectively.  Does the sample distribution agree with what you would expect?

</div>

In [ ]:
# EXERCISE 3
cudaq.set_target("density-matrix-cpu")

noise = cudaq.NoiseModel()
noise.add_channel('x', [0], cudaq.BitFlipChannel(.1))
noise.add_channel('x', [1], cudaq.BitFlipChannel(.25))


@cudaq.kernel
def test():
    reg = cudaq.qvector(2)
    x(reg)


print(cudaq.sample(test, noise_model=noise))

---

## 3.2 Two Ways to Simulate Noise

**Density matrix simulation** can produce exact results with the quality of simulation limited only by the accuracy of the underlying noise model.  Unfortunately, **density matrix simulation** is expensive and requires storage of the entire $2^N \times 2^N $ matrix, limiting it to a smaller number of qubits. 

This scalability problem can be circumvented with a method called **trajectory simulation** which allows for approximate noise simulation at much larger scales.  In trajectory simulation, a statevector is evolved through the circuit and at each noisy gate, a Kraus operator is selected at random based on its probability. This is sometimes called Monte Carlo trajectory simulation.  This produces one trajectory per shot, and in the limit of many trajectories, the results converge to those of the density matrix simulation.  The CUDA-Q density matrix simulator and GPU-accelerated statevector simulator can each produce equivalent results in noise simulations.  Run the density matrix version below and look at the output.

In [ ]:
cudaq.set_target("density-matrix-cpu")

noise = cudaq.NoiseModel()
for q in range(3):
    noise.add_channel('x', [q], cudaq.BitFlipChannel(.2))

@cudaq.kernel
def test():
    reg = cudaq.qvector(3)
    x(reg)


cudaq.set_noise(noise)
for x in range(2):
    print(cudaq.get_state(test))


print(cudaq.sample(test, noise_model = noise))

**Trajectory simulation** can run in CUDA-Q by simply changing the target to `nvidia`.  If the kernel below had no noise, the statevector (output from `get_state`) should be [0,0,0,0,0,0,0,1] corresponding to the $\ket{111}$ state. When sampling is performed with the trajectory based simulator, the Kraus operators are applied based on their probabilities to produce a new state vector for each shot. The widget below allows you to explore the possible outcomes and their associated probabilities.   

Try running the CUDA-Q simulation above with two or three different bitflip error probabilities and use the [interactive trajectory noise widget](https://nvidia.github.io/cuda-q-academic/interactive_widgets/trajectory_noise_demo.html) to confirm that the results from the density matrix simulations above match the expected distribution from the trajectory-based approach.

Running the code below, notice `get_state` produces a different state vector each time. Because the number of possible trajectories is small, trajectory based sampling can reproduce the same distribution that would be obtained from density matrix simulation.

In [ ]:
cudaq.set_target("nvidia")

noise = cudaq.NoiseModel()
for q in range(3):
    noise.add_channel('x', [q], cudaq.BitFlipChannel(.2))

@cudaq.kernel
def test():
    reg = cudaq.qvector(3)
    x(reg)

cudaq.set_noise(noise)
for x in range(20):
    print(cudaq.get_state(test))

print(cudaq.sample(test))

Another benefit of trajectory based simulation is that it can be used with tensor network based simulators to simulate circuits that would be far too large for density matrix or statevector simulation. CUDA-Q can run exact tensor network or approximate Matrix Product State (MPS) simulations with trajectory based simulation to simulate systems of hundreds to thousands of qubits.

Clever sampling algorithms can also be used to filter trajectories and exclude certain types of errors or focus on sampling a particular type of error. This technique is called importance sampling and is another active area of research for making noisy simulation more practical and beneficial to QEC researchers.

---

## 3.3 Use Cases for Noisy Simulations

This section will explore three use cases of noisy simulation used to model the impact of noise patterns on algorithms, perform **quantum error mitigation**, and run QEC experiments.

### 3.3a: Understanding How Noise Impacts Algorithm Results

A natural application of noisy simulation is to explore how different noise patterns might impact the results of an algorithm.  Such simulations can be beneficial for a number of reasons. This section along with the following two will explore three use cases for noisy circuit simulation. 

The first use case is to better understand how device noise impacts the outcome of an algorithm. Researchers can use these sorts of results to produce insights that guide compilation methods by identifying how particular algorithms might be more or less sensitive to particular noise channels. Such an approach is also useful to develop noise models by tuning them to agree with the results obtained running the same application experimentally.

This section will walk you through an exercise to explore the impact of noise on a standard chemistry experiment.  The CUDA-Q Solvers library makes it easy to prepare a quantum circuit to compute the ground state energy of a molecule. The code section below prepares a circuit with the standard UCCSD ansatz as well as the Hamiltonian of the hydrogen molecule. Run the cell below to get the noiseless energy and see a print out of the circuit.  Note: the circuit parameters are not optimized, but this does not matter as it is just a reference point to study the impact of noise. 

In [ ]:
cudaq.set_target("nvidia")
print(cudaq.sample(test, noise_model = noise))


geometry = [('H', (0., 0., 0.)), ('H', (0., 0., .7474))]
molecule = solvers.create_molecule(geometry, 'sto-3g', 0, 0)

noise_empty = cudaq.NoiseModel()

numQubits = molecule.n_orbitals * 2
numElectrons = molecule.n_electrons
mol_spin = 0
initialX = [-.2] * solvers.stateprep.get_num_uccsd_parameters(
    numElectrons, numQubits)

@cudaq.kernel
def uccsd():
    reg = cudaq.qvector(numQubits)
    for i in range(2):
        x(reg[i])
    solvers.stateprep.uccsd(reg, initialX, numElectrons, mol_spin)

#noiseless value
print(cudaq.observe(uccsd, molecule.hamiltonian, noise_model = noise_empty).expectation())
noiseless = cudaq.observe(uccsd, molecule.hamiltonian, noise_model = noise_empty).expectation()

print(cudaq.draw(uccsd))

As a bonus, try repeating this exercise using a simulator tuned to mimic the noise of a physical QPU.  CUDA-Q provides access to, for example, the [Quantinuum H-2 emulator](https://nvidia.github.io/cuda-quantum/latest/using/backends/hardware/iontrap.html#quantinuum),  [IonQ's emulator](https://nvidia.github.io/cuda-quantum/latest/using/backends/hardware/iontrap.html#ionq), [Infleqtion's noisy simulator](https://nvidia.github.io/cuda-quantum/latest/using/backends/hardware/neutralatom.html#infleqtion), and more.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 4:</span>**

Now, write a function that computes the expectation values for various configurations of errors.  The function comments will guide you on the inputs and what the function should return.

</div>

In [ ]:
# EXERCISE 4
def get_noisy_data(e_type =[], gate=[] , qubit=[], prob=[], shots=-1, trajectories=None):
    """The function takes in various configurations of noise channels, builds a noise model, uses the noise model to obtain 40 expectation values,
       and the returns a list of the difference between the noisy expectation values and the noiseless.

    Parameters
    ----------
    e_type: list[int]
        List designating the type of each error applied for a given noise channel.
            1=phaseflip
            2=amplitude damping
            3=depolarization
    gate: list[str]
        List designating the type of gate each error is applied on (e.g. 'h' or'x').
    qubit: list[int]
        List designating the qubit index where the noise channel is applied.
    prob: list[float]
        List designates the probability of error for each noise channel
    shots: int
        Designates the number of shots used to compute the expectation value.  Default (-1) is exact, non-shot based result.
    trajectories: int
        Designates the number of trajecttores sampled when computing the expectation value.

    Returns
    -------
    list[float]
        List of length 40 where each elementis the diffence between the noisy and noiseless value.
    """

    noise = cudaq.NoiseModel()

    for x in range(len(e_type)):
        
        if e_type[x] == 0:
            noise.add_channel(gate[x], [qubit[x]], cudaq.BitFlipChannel(prob[x]))

        if e_type[x] == 1:
            noise.add_channel(gate[x], [qubit[x]], cudaq.PhaseFlipChannel(prob[x]))

        if e_type[x] == 2:
            noise.add_channel(gate[x], [qubit[x]], cudaq.AmplitudeDampingChannel(prob[x]))

        if e_type[x] == 3:
            noise.add_channel(gate[x], [qubit[x]], cudaq.DepolarizationChannel(prob[x]))
            
    data =[]
    
    for x in range(40):
        data.append(cudaq.observe(uccsd, molecule.hamiltonian, shots_count=shots, noise_model=noise, num_trajectories=trajectories).expectation())

    normalized_data = [x - noiseless for x in data]
    return normalized_data

The function below, will take the result from `get_noisy_data` and plot them.  Just enter each output from `get_noisy_data` as an element of the `datasets` list variable, name the categories for the x-axis, and label the datasets if more than one is provided. The next cell provides an example.

In [ ]:
def plot_data(datasets, categories, labels=None):
    """
    Plots the mean and ±1 SD error bars for one or more datasets,
    with the y-axis centered at 0 for each category.
    
    Parameters:
    -----------
    datasets : list of lists (single dataset) or list of list of lists (multiple datasets)
        Each sub-list (or sub-sub-list if multiple) contains numeric values for a given category.
    categories : list of str
        Labels for the categories corresponding to each data sub-list.
    labels : list of str, optional
        Labels for each dataset if multiple datasets are provided.
    """
    # If only a single dataset (list of lists) is given, wrap it so we can loop uniformly
    if not isinstance(datasets[0][0], (list, np.ndarray)):
        datasets = [datasets]
    
    # If no labels given, auto-generate labels
    if labels is None:
        labels = [f"Dataset {i+1}" for i in range(len(datasets))]

    plt.figure()
    all_vals = []  # Collect all mean±std values to compute symmetric y-axis

    # Plot each dataset with a small horizontal offset
    for i, data in enumerate(datasets):
        means = [np.mean(vals) for vals in data]
        stds = [np.std(vals) for vals in data]
        x_positions = np.arange(len(categories)) + i * 0.1

        plt.errorbar(
            x_positions, means, yerr=stds, fmt='o', capsize=5, 
            label=labels[i]
        )

        # Store values for y-axis range calculation
        all_vals.extend([m + s for m, s in zip(means, stds)])
        all_vals.extend([m - s for m, s in zip(means, stds)])

    # Center x-ticks between all plotted points
    midpoint_offset = ((len(datasets) - 1) * 0.1) / 2
    plt.xticks(np.arange(len(categories)) + midpoint_offset, categories)
    
    plt.ylabel('Mean Value (Expectation Value)')
    plt.title('Mean ±1 SD Error Bars (Centered at 0 = No Noise)')

    # Compute symmetric y-limits around 0
    y_upper = max(all_vals)
    y_lower = min(all_vals)
    max_range = max(abs(y_upper), abs(y_lower))
    plt.ylim([-max_range, max_range])

    plt.axhline(0, color='black', linewidth=0.8, linestyle='--')  # Reference line at y=0

    if len(datasets) > 1:
        plt.legend()

    plt.show()


#### Analyzing Shot Number ####

A source of noise not discussed thus far, and of a completely different nature than any physical noise channel, is sampling error. Sampling based quantum algorithms can produce inaccurate results simply due to sampling error, even if the hardware were perfect.  The code below, demonstrates how to use the `plot_data` function, and produces the distributions of hydrogen ground state energies obtained with anywhere from 10 to 10000 shots compared to 0 which is the noiseless result in the limit of infinite samples.

Notice how sampling based error is centered on the noiseless result and rapidly dissipates as more shots are used.   The rest of the simulations below will not perform shot based sampling so the only deviations from zero are due to the noise channels you implement below. Nevertheless, it is important to not forget that sampling error is a ubiquitous source of error for most quantum algorithms.

In [ ]:
data = [get_noisy_data(shots=10),
        get_noisy_data(shots=100),
        get_noisy_data(shots=1000),
        get_noisy_data(shots=10000),
]

categories = ['10','100', '1000', '10000' ] 
labels =['series 1']

plot_data([data], categories, labels)



#### Analyzing Trajectory Number ####

Similar to sampling error, trajectory based noise simulation is also dependent on the number of trajectories used to compute the expectation value. Even if each trajectory is used to compute the expectation value exactly, if too few statevectors are sampled, the results of the noisy simulations can become unreliable. 

Build a data set that only applies bitflip errors on $X$ gates for every qubit with probability 0.01. Vary the trajectories from 10 to 10000 and comment on the results.  

Does this noise model systematically over or under estimate the energy prediction? Could you be confident in this observation if you only used 10 trajectories?

In [ ]:
data = [
    get_noisy_data([0]*4, ['x']*4, [0,1,2,3], [0.01]*4, trajectories=10),
    get_noisy_data([0]*4, ['x']*4, [0,1,2,3], [0.01]*4, trajectories=100),
    get_noisy_data([0]*4, ['x']*4, [0,1,2,3], [0.01]*4, trajectories=1000),
    get_noisy_data([0]*4, ['x']*4, [0,1,2,3], [0.01]*4, trajectories=10000),
]

categories = ['10', '100', '1000', '10000']

plot_data([data], categories)


#The bitflip errors seems to systematically overestimate the energy result.
#In the first case with only 10 trajectories sampled, an error of 0 is within the confidence interval so this conclusion cannot be drawn with certainty.
#As the number of trajectories increaces, the variability in results becomes very small.

#### Analyzing Error Probability and Gate Type ####

Now, turning to questions more specific to the structure of this algorithm's circuit, one can ask how changing the probability of a bitflip error impacts the computed energy. Plot the results corresponding to bitflip errors on all $X$ gates for all qubits with decreasing probabilities of .1, .01, .001, .0001. Plot a second series on the same graph but this time have the bitflip error applied on the $H$ gates.


For which type of gate are bitflip errors more problematic to the result?

In [ ]:
datax = [get_noisy_data([0]*4,['x']*4,[0,1,2,3],[.1,.1,.1,.1]),
        get_noisy_data([0]*4,['x']*4,[0,1,2,3],[.01,.01,.01,.01]),
        get_noisy_data([0]*4,['x']*4,[0,1,2,3],[.001,.001,.001,.001]),
        get_noisy_data([0]*4,['x']*4,[0,1,2,3],[.0001,.0001,.0001,.0001]),
]

datah = [get_noisy_data([0]*4,['h']*4,[0,1,2,3],[.1,.1,.1,.1]),
        get_noisy_data([0]*4,['h']*4,[0,1,2,3],[.01,.01,.01,.01]),
        get_noisy_data([0]*4,['h']*4,[0,1,2,3],[.001,.001,.001,.001]),
        get_noisy_data([0]*4,['h']*4,[0,1,2,3],[.0001,.0001,.0001,.0001]),
]

categories = ['p=.1','p=.01', 'p=.001', 'p=.0001' ] 

plot_data([datax,datah], categories, labels=['errors on x gate', 'errors on h gate'])

#Bitflip errors seem to be more problematic when they occur on Hadamard gates rather than
#X gates and the difference is more pronounced at high error rates.

#### Analyzing Error Type ####

Now, fix the error probability at 0.1 and this time vary the type of error.  Keep the two series with errors occurring on $X$ and $H$ gates.

Which type of error is most severe? Are there any errors that have little to no impact?

In [ ]:
datax = [get_noisy_data([0]*4,['x']*4,[0,1,2,3],[.1]*4),
         get_noisy_data([1]*4,['x']*4,[0,1,2,3],[.1]*4),
         get_noisy_data([2]*4,['x']*4,[0,1,2,3],[.1]*4),
         get_noisy_data([3]*4,['x']*4,[0,1,2,3],[.1]*4)]

datah = [get_noisy_data([0]*4,['h']*4,[0,1,2,3],[.1]*4),
         get_noisy_data([1]*4,['h']*4,[0,1,2,3],[.1]*4),
         get_noisy_data([2]*4,['h']*4,[0,1,2,3],[.1]*4),
         get_noisy_data([3]*4,['h']*4,[0,1,2,3],[.1]*4)]


categories = ['bitflip','phaseflip', 'amplitude damping', 'depolarization'] 

plot_data([datax,datah], categories, labels=['errors on x gate', 'errors on h gate'])

#Depolarization errors on Hadamard gates are the most problematic
#while phaseflip errors seem to have no impact to the predicted energy when they occur on X gates.

#### Analyzing Error Location (Qubit) ####

Finally, keep the two series and now perform only Amplitude Damping errors. This time, set all of the probabilities equal to 0, expect for the single error prone qubit which is set to a probability of 0.1.  

If you had a QPU where you knew qubit A was particularly prone to amplitude damping for some reason, when you compile the algorithm, which wire of the quantum circuit (q0, q1, q2, q3) should map to qubit A to ensure the best results based on your simulations?

In [ ]:
data = [get_noisy_data([2]*4,['x']*4,[0,1,2,3],[.1,0,0,0]),
         get_noisy_data([2]*4,['x']*4,[0,1,2,3],[0,.1,0,0]),
         get_noisy_data([2]*4,['x']*4,[0,1,2,3],[0,0,.1,0]),
         get_noisy_data([2]*4,['x']*4,[0,1,2,3],[0,0,0,.1])]

datap = [get_noisy_data([2]*4,['h']*4,[0,1,2,3],[.1,0,0,0]),
         get_noisy_data([2]*4,['h']*4,[0,1,2,3],[0,.1,0,0]),
         get_noisy_data([2]*4,['h']*4,[0,1,2,3],[0,0,.1,0]),
         get_noisy_data([2]*4,['h']*4,[0,1,2,3],[0,0,0,.1])]



categories = ['q0','q1', 'q2', 'q3'] 

plot_data([data,datap], categories, labels=['errors on x gate', 'errors on h gate'])


#Amplitude damping errors seem to have less impact when they ocure on q2 or q3, thus,
#if qubit A is prone to these errors, it should map to q2 or q3. 

### 3.3b: Zero Noise Extrapolation

QPU results today are sometimes improved using quantum error mitigation (QEM) techniques.  QEM techniques use classical postprocessing to improve results without the utilization of proper QEC protocols. One such QEM technique is **zero noise extrapolation (ZNE)**.  The idea behind ZNE is that it is really hard to remove noise from an algorithm run on a physical QPU, but it is very easy to add noise. 

The ZNE process works by applying increasing factors of error through clever application of the identity operator. For example, consider a circuit composed of a single layer of $R_X$ rotations of $\pi$ radians. Applying the same gate three times is mathematically the same as applying it once and should have no impact on the result. 

$$R_X(\pi)R_X(\pi)R_X(\pi) = R_X(\pi)I = R_X(\pi)$$

Experimentally, this is truly the identity operation as each gate is a noise channel and the total noise factor is increased from 1x to 3x.  If this procedure is continued (5x, 7x, 9x, ...) the data can be fit to a curve and extrapolated back to estimate the experimentally inaccessible case of a 0x noise factor!  So, paradoxically, adding noise can improve the result. 

<img src="../Images/noisy/zneplot.png" alt="Plot showing expectation values versus noise factor with measured data points at 1x, 3x, 5x, 7x, and 9x noise, a polynomial fit curve, and the extrapolated zero-noise value at the y-intercept" style="width: 1000px;"/>


ZNE is a useful technique because it can be used experimentally. Noisy circuit simulation can demonstrate its effectiveness and help benchmark the effectiveness of ZNE when used on a physical QPU, help refine noise models, and test other QEM techniques before running experiments.  


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 5:</span>**

You will now code a ZNE example by following the steps below:

1. Create a Random Hamiltonian for a larger (20 qubit circuit)
2. Define a quantum circuit with a layer of $R_x(\pi/2)$ gates followed by a layer of $X$ gates.
3. Put a bitflip error on the $X$ gates and an **amplitude damping** error on the $R_X$ gates.
4. Perform ZNE to obtain a correction for each. (Hint: use the `np.poly1d()` to fit a polynomial.)
5. Apply the correction to the original noisy circuit and calculate the percent error of the noisy circuit and the ZNE corrected result relative to the noiseless case.

</div>

In [ ]:
# EXERCISE 5
hamiltonian = 0

for i in range(20):
    hamiltonian += np.random.uniform(-1, 1)*spin.z(i)
    hamiltonian += np.random.uniform(-1, 1)*spin.y(i)
    hamiltonian += np.random.uniform(-1, 1)*spin.x(i)


error_prob = 0.1

noise = cudaq.NoiseModel()
for i in range(20):
    noise.add_channel('x', [i], cudaq.BitFlipChannel(error_prob))
    noise.add_channel('rx', [i], cudaq.AmplitudeDampingChannel(error_prob))

cudaq.set_noise(noise)

@cudaq.kernel
def zne(factor: int):
    reg = cudaq.qvector(20)

    for i in range(factor):
        x(reg)
        rx(np.pi,reg)


factors = [1,3,5,7,9]
results = []
for i in range (5):
    results.append(cudaq.observe(zne, hamiltonian,2*i+1).expectation()) 

cudaq.unset_noise() 
noiseless = cudaq.observe(zne, hamiltonian,1).expectation()

linear = np.poly1d(np.polyfit(factors, results, 1))
quadratic = np.poly1d(np.polyfit(factors, results, 2))

def plot_zero_noise_extrapolation(noise_factors, measurements, poly_fit):
    """
    Plot the original data vs. noise factor and the polynomial fit extended 
    down to noise=0 to show the extrapolation result.
    """
    # Create a range of noise values from 0 to slightly beyond the largest noise factor
    x_range = np.linspace(0, max(noise_factors) + 0.5, 50)
    y_fit = poly_fit(x_range)

    # Plot measured data points
    plt.scatter(noise_factors, measurements, label='Measured Data', color='blue')
    # Plot polynomial fit
    plt.plot(x_range, y_fit, label='Fit (degree = {})'.format(poly_fit.order), color='red')

    # Highlight the zero-noise extrapolation point
    extrapolated_value = poly_fit(0)
    plt.scatter([0], [extrapolated_value], color='green', zorder=5, 
                label='Zero-Noise Extrapolation = {:.3f}'.format(extrapolated_value))
    
    plt.xlabel('Noise Factor')
    plt.ylabel('Measured Expectation Value')
    plt.title('Zero-Noise Extrapolation')
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    plt.legend()
    plt.grid(True)
    plt.show()
    print(f"Percent Error of ZNE Estimate {(extrapolated_value - noiseless)/noiseless*100} %")


print(f"Percent Error of Uncorrected Noisy Circuit: {(results[0] - noiseless)/noiseless*100} %")


plot_zero_noise_extrapolation(factors, results, linear)
plot_zero_noise_extrapolation(factors, results, quadratic)


### 3.3c: QEC Experiments

Noisy circuit simulation is perhaps most useful as a tool for QEC researchers.  One can test how a code will perform in a variety of different noise conditions.  Assuming an accurate noise model, this can be a great way to assess characteristics of new codes. Below you will add noise to the Steane code you prepared in lab 2. 


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 6:</span>**

Apply noise to the Steane code in the following three ways and determine which case produces the best and worst logical error rates, keeping the probability of error fixed at 0.05.  In which cases is the logical error rate an improvement over the 0.05 error rate?

1. Use `cudaq.apply_noise(cudaq.XError, p, data_qubits[j])` to manually apply Kraus operators following encoding of the Steane code but before the stabilizer checks are run.  These errors are not tied to gates but model errors induced while the system idles.
2. Now, use `cudaq.apply_noise(cudaq.Depolarization2, p, data_qubits[i], data_qubits[j])` to apply a depolarization error following all of the two qubit gates in the encoding circuit, where q and r are the two qubits involved in the gate operation.
3. Apply a bitflip noise channel to all `mz` measurements.  In this case, errors are also possible in measurements performed on the ancillas.  This helps model situations where measurements are performed in a way that is not fault tolerant.

</div>

In [ ]:
# EXERCISE 6
cudaq.set_target('stim')

p = 0.05
cudaq.unset_noise()
noise = cudaq.NoiseModel()

@cudaq.kernel
def steane_code():
    """Prepares a kernel for the Steane Code
    Returns
    -------
    cudaq.kernel
        Kernel for running the Steane code
    """   
    data_qubits = cudaq.qvector(7)
    ancilla_qubits = cudaq.qvector(3)

    # Create a superposition over all possible combinations of parity check bits
    h(data_qubits[4])
    h(data_qubits[5])
    h(data_qubits[6])

    #Entangle states to enforce constraints of parity check matrix

    x.ctrl(data_qubits[0],data_qubits[1])
    x.ctrl(data_qubits[0],data_qubits[2])
    x.ctrl(data_qubits[4],data_qubits[0])
    x.ctrl(data_qubits[4],data_qubits[1])
    x.ctrl(data_qubits[4],data_qubits[3])

    x.ctrl(data_qubits[5],data_qubits[0])
    x.ctrl(data_qubits[5],data_qubits[2])
    x.ctrl(data_qubits[5],data_qubits[3])

    x.ctrl(data_qubits[6],data_qubits[1])
    x.ctrl(data_qubits[6],data_qubits[2])
    x.ctrl(data_qubits[6],data_qubits[3])

    # Detect Z errors
    h(ancilla_qubits)

    x.ctrl(ancilla_qubits[0],data_qubits[0])
    x.ctrl(ancilla_qubits[0],data_qubits[1])
    x.ctrl(ancilla_qubits[0],data_qubits[3])
    x.ctrl(ancilla_qubits[0],data_qubits[4])

    x.ctrl(ancilla_qubits[1],data_qubits[0])
    x.ctrl(ancilla_qubits[1],data_qubits[2])
    x.ctrl(ancilla_qubits[1],data_qubits[3])
    x.ctrl(ancilla_qubits[1],data_qubits[5])

    x.ctrl(ancilla_qubits[2],data_qubits[1])
    x.ctrl(ancilla_qubits[2],data_qubits[2])
    x.ctrl(ancilla_qubits[2],data_qubits[3])
    x.ctrl(ancilla_qubits[2],data_qubits[6])

    h(ancilla_qubits)

    sz1 = mz(ancilla_qubits[0])
    sz2 = mz(ancilla_qubits[1])
    sz3 = mz(ancilla_qubits[2])

    #Reset ancillas
    reset(ancilla_qubits)

    # Detect X errors
    h(ancilla_qubits)

    z.ctrl(ancilla_qubits[0],data_qubits[0])
    z.ctrl(ancilla_qubits[0],data_qubits[1])
    z.ctrl(ancilla_qubits[0],data_qubits[3])
    z.ctrl(ancilla_qubits[0],data_qubits[4])

    z.ctrl(ancilla_qubits[1],data_qubits[0])
    z.ctrl(ancilla_qubits[1],data_qubits[2])
    z.ctrl(ancilla_qubits[1],data_qubits[3])
    z.ctrl(ancilla_qubits[1],data_qubits[5])

    z.ctrl(ancilla_qubits[2],data_qubits[1])
    z.ctrl(ancilla_qubits[2],data_qubits[2])
    z.ctrl(ancilla_qubits[2],data_qubits[3])
    z.ctrl(ancilla_qubits[2],data_qubits[6])

    h(ancilla_qubits)

    sx1 = mz(ancilla_qubits[0])
    sx2 = mz(ancilla_qubits[1])
    sx3 = mz(ancilla_qubits[2])


    # Correct X errors

    if sx1 and sx2 and sx3:
        x(data_qubits[3])
    elif sx1 and sx2:
        x(data_qubits[0])
    elif sx1 and sx3:
        x(data_qubits[1])
    elif sx2 and sx3:
        x(data_qubits[2])
    elif sx1:
        x(data_qubits[4])
    elif sx2:
        x(data_qubits[5])
    elif sx3:
        x(data_qubits[6])



    # Correct Z errors

    if sz1 and sz2 and sz3:
        z(data_qubits[3])
    elif sz1 and sz2:
        z(data_qubits[0])
    elif sz1 and sz3:
        z(data_qubits[1])
    elif sz2 and sz3:
        z(data_qubits[2])
    elif sz1:
        z(data_qubits[4])
    elif sz2:
        z(data_qubits[5])
    elif sz3:
        z(data_qubits[6])

    data_qubits



results = cudaq.sample(steane_code, shots_count=10000, noise_model=noise)

print(results)

ones = 0
zeros = 0 
for bitstring in results:
    counts = results.count(bitstring)
    parity = sum(int(bit) for bit in bitstring[0:7]) % 2
    
    if parity == 0:
        zeros += 1*counts
    else:
        ones += 1*counts

logical_rate = ones/(zeros+ones)
            
print(f"logical error rate:{logical_rate}")


In [ ]:
"""
The different error sources should get increasingly worse, with the worst case involving measurement which cannot be fixed by stabilizer measurements.

Case 1 is performed by adding the following code right after the encoding circuit and should produce a result around 0.04.


    for j in range(7):
            cudaq.apply_noise(cudaq.XError, p, data_qubits[j])
"""

"""
**Answer:**  
Case 2 is performed by adding individual errors following each gate in the encoding circuit. See example below.
The logical error rate should be around 0.11

    x.ctrl(data_qubits[0],data_qubits[1])
    cudaq.apply_noise(cudaq.Depolarization2, p, data_qubits[0], data_qubits[1])
"""


"""
Case 3 is performed by adding the following line before the CUDA-Q kernel.  This should produce a logical error rate of around 0.45. 

    noise = cudaq.NoiseModel()
    noise.add_all_qubit_channel("mz", cudaq.BitFlipChannel(0.1))
"""


---

## 3.4 Using Dynamical Simulations to Build a Noise Model

The noise models used thus far are meant to mimic the underlying physics of physical qubits. Often, noise models are heavily informed by experiment, but extracting meaningful insights can be extremely difficult for such complex systems. 

<img src="../Images/noisy/dynamics_noise_model.png" alt="Flowchart showing the process of building a noise model: dynamical simulation of qubit physics produces decoherence parameters that are used to parameterize noise channels for gate-level circuit simulation" style="width: 1200px;"/>

To help with this task, the physics of the qubits can also be simulated to better understand noise sources and improve interpretation of experimental data.  This sort of simulation is known as **dynamical simulation** and models the evolution of a quantum system over time as the system interacts with its environment. 

The code below will help you walk through an example of using dynamical simulation to produce a noise model for a single qubit **amplitude damping** channel. Recall, the corresponding noise channel looks like this. 

$$ \epsilon(\rho)  = \sqrt{1-p} \cdot \rho + \sqrt{p} \cdot \rho \cdot 0.5 \cdot (X+iY) $$

Thus, the goal is to simulate a simple qubit system to determine what $p$, the probability of energy loss resulting in decay to the ground state, is.

Dynamical simulation is its own topic that warrants a detailed introduction that will not be provided here.  Instead, the steps of the dynamical simulation will be discussed at a high level while curious readers can explore the CUDA-Q dynamics page for more information and more detailed examples.

To get started, this example will use the CUDA-Q dynamics backend, set like any other backend.

In [ ]:
cudaq.set_target("dynamics")

You will simulate a superconducting transmon qubit which is driven close to resonance, to produce so called Rabi oscillations. You will model a qubit which has a set resonant frequency, add a driving term that depends on time and will drive the system close to its resonant frequency, and by doing so, introduce a change in population from the $\ket{0}$ state to the  $\ket{1}$ state.  In other words, you will simulate thr underlying physics required to perform an $X$ gate.

The first step is to construct the Hamiltonian of the system. The Hamiltonian consists of a $Z$ term that encodes the resonant frequency for the qubit corresponding to the transition from the $\ket{0}$ to the $\ket{1}$ state. The second, is a driving term that applies a time dependent function  towards the qubits resonant frequency.  When this happens, Rabi oscillations occur and the population of the system changes from 100\% $\ket{0}$ to 100\% $\ket{1}$, that is, an $X$ gate applied to the qubit!


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px;">
    <h3 style="color: #76b900; margin-top: 0;"> Exercise 7 :</h3>
    <p style="font-size: 16px; color: #333;">
The code below sets up the the problem Hamiltonian, defines the dimensions of the system and specifies the initial ground state. The terms have more meaning than described above, but their details are not relevant for the purposes of this exercise.
    </p>
</div>



In [ ]:
omega_z = 10.0 * 2 * np.pi
omega_x = 2 * np.pi
omega_drive = 1 * omega_z

hamiltonian = 0.5 * omega_z * spin.z(0)
hamiltonian += omega_x * ScalarOperator(lambda t: np.cos(omega_drive * t)) * spin.x(0)

dimensions = {0: 2}

rho0 = cudaq.State.from_data(
    cp.array([[1.0, 0.0], [0.0, 0.0]], dtype=cp.complex128))

Dynamics simulations are performed numerically and require a time step specifying how the evolution operator is applied. For this problem it is setup below. 

In [ ]:
t_final = np.pi / omega_x
dt = 2.0 * np.pi / omega_drive / 100
n_steps = int(np.ceil(t_final / dt)) + 1
steps = np.linspace(0, t_final, n_steps)
schedule = Schedule(steps, ["t"])

You are now ready to run the simulation using CUDA-Q's `evolve` function, which does require GPU access. The code cell below shows the baseline case where the qubit does not interact with its environment (i.e. there is no decoherence), and the plot shows a the probability of sampling the $\ket{1}$ state rise from 0 to 1.  This corresponds to a pulse used to implement a perfect noiseless $X$ gate. 

In [ ]:
evolution_result = cudaq.evolve(hamiltonian,
                                dimensions,
                                schedule,
                                rho0,
                                observables=[operators.number(0)],
                                collapse_operators=[],
                                store_intermediate_results=True,
                                integrator=ScipyZvodeIntegrator())

get_result = lambda idx, res: [
    exp_vals[idx].expectation() for exp_vals in res.expectation_values()
]
ideal_results = [
    get_result(0, evolution_result)]



plt.figure()
plt.plot(steps, ideal_results[0])
plt.ylabel("Probability(1)")
plt.xlabel("Time")
plt.title("No Decoherence")

Now, things can get a bit more interesting when decoherence is factored in.  In this case, a so called collapse operator is added to the dynamical simulation to model the decay of energy into the environment as the system evolves. In this case the simple collapse operator `np.sqrt(gamma_sm) * spin.minus(0)` is added.  `gamma_sm` is set to 1, but would in practice be determined by some experimental quantity.  Now, by running this simulation, you will be able to simulate amplitude damping and see what the probability of the $\ket{1}$ state remains after the pulse. 

In [ ]:
gamma_sm = 1.0
evolution_result_decay = cudaq.evolve(
    hamiltonian,
    dimensions,
    schedule,
    rho0,
    observables=[operators.number(0)],
    collapse_operators=[
        np.sqrt(gamma_sm) * spin.minus(0),
    ],
    store_intermediate_results=True,
    integrator=ScipyZvodeIntegrator())

decoherence_results = [
    get_result(0, evolution_result_decay)
]

plt.figure()
plt.plot(steps, decoherence_results[0])
plt.ylabel("Probability(1)")
plt.xlabel("Time")
plt.title("Decoherence")

The pulse now has a peak of around .85, meaning $p=.15$ is a reasonable choice to parameterize an amplitude damping channel.

In [ ]:
cudaq.set_target('density-matrix-cpu')
cudaq.set_random_seed(13)
noise = cudaq.NoiseModel()
 
amplitude_damping = cudaq.AmplitudeDampingChannel(0.15) # this value is from the dynamics simulation
noise.add_channel('x', [0], amplitude_damping)

kernel = cudaq.make_kernel()
qubit = kernel.qalloc()
kernel.x(qubit)
kernel.mz(qubit)
counts = cudaq.sample(kernel, noise_model=noise)
prob_1 = counts.probability("1")
print("Probability of |1> from the gate-level simulation with noise:", prob_1)

cudaq.reset_target()

This is a very simple example, and in practice it is much harder to derive noise models from dynamical simulations.  Nevertheless, they are powerful tools for understanding noise.  You can also tweak other aspects of the simulation for more complex situations.  For example, try the following and see how they might change the amplitude damping parameterization.

1. change `omega_drive = 0.95 * omega_z` to be close to but not the same as the resonance frequency.
2. Add 0.1 to `t_final = np.pi / omega_x` to test what would happen if a gate pulse is applied for too long.

In [ ]:
"""
Using an off-resonance frequency means there is not hte same energy transfer and the peak only goes up to around 0.8, even without decoherence.
Changing the time range means that the pulse no longer stops at its peak and has started to slightly decay. 
"""

## Conclusion

Noise is the greatest challenge facing quantum computers.  Accurate simulations can help us understand both the sources and impacts of noise to guide development of better hardware, algorithms, and QEC codes.   You now know how to utilize CUDA-Q for noise modeling as well as a number of situations where noise modeling is useful.  Scaling up any of these examples makes simulation much more challenging and requires the power of CUDA-Q and AI supercomputing to usher in new advancements to the field.

**Related Notebooks:**
* [QEC Stabilizers](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/02_QEC_Stabilizers.ipynb) — covers the Steane code used in this notebook’s QEC exercises
* [QEC Decoders](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/04_QEC_Decoders.ipynb) — next notebook in the QEC101 series
* [VQE and GQE](https://github.com/NVIDIA/cuda-q-academic/blob/main/chemistry-simulations/vqe_and_gqe.ipynb) — explores the variational chemistry algorithm used in the noise impact analysis